# 带吃水限制的旅行商问题 (TSPDL)

**类别：** 路径规划

来源: [https://www.hexaly.com/templates/traveling-salesman-problem-with-draft-limits-tspdl](https://www.hexaly.com/templates/traveling-salesman-problem-with-draft-limits-tspdl)


## 问题

**带吃水限制的旅行商问题 (TSPDL)** 是标准 TSP 的一个变体，出现在海上运输的背景下。给定 n 个港口以及每对港口之间的距离，并考虑每个港口入口处对最大允许吃水（船体水面线与船底之间的垂直距离）的限制，寻找一条总长度最短的环游路径，使其恰好访问每个港口一次。从港口 i 到港口 j 的距离与从港口 j 到港口 i 的距离可能不同。

	

### 学到的建模原则

- 添加 [list 决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来建模城市的排列
- 定义 [lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html) 来计算行驶距离
- 使用 [递归 lambda 函数](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 定义一个数组，以计算船舶在路径上的重量


## 数据

所提供的带吃水限制的旅行商问题 (TSPDL) 实例改编自 [TSPLib](http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/) 非对称 TSP 数据库，采用 TSPLib 显式格式：

- 城市数量在关键字 “N” 之后给出。
- 完整的距离矩阵在关键字 “Distance” 之后给出。
- 每个港口的需求量在关键字 “Demand” 之后给出。
- 每个港口的最大吃水深度在关键字 “Draft” 之后给出。


## 程序

带吃水限制的旅行商问题 (TSPDL) 的 Hexaly 模型是 TSP 模型的扩展。问题中路径规划部分的细节（表示路径的 list 决策变量以及总行驶距离的计算）请参考该模型。

船舶的吃水是指水面线与船底之间的距离。吃水深度随着船舶的载重增加而增加，每个港口都有吃水限制，超过该限制的船舶无法进入该港。因此，在海运中，能否访问某个地点取决于所装载货物的多少。在其他场景中，只要某个地点的可访问性取决于车辆重量，也会出现类似的限制。与取货送货问题 (PDP) 类似，这需要通过 [**递归数组**](https://www.hexaly.com/docs/last/mathematicaloperators/delegates.html#special-case) 来计算车辆在路径上的重量：访问一个城市之后的重量等于访问前一个城市之后的重量减去当前城市的需求量。然后我们使用可变参数 ‘and（与）’ 运算符来确保所有吃水限制都得到满足。

另一种做法是，你可以最小化累计的超重：

overweight <- sum(0...nbCities, i => max(0, weight[i] - weightLimit[cities[i]]));

当寻找满足所有吃水限制的解需要数秒以上时，这种方法会是一个不错的方案。


## Python 实现


In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import sys

if len(sys.argv) < 2:
    print("Usage: python tspdl.py inputFile [outputFile] [timeLimit]")
    sys.exit(1)


def read_elem(filename):
    with open(filename) as f:
        return [str(elem) for elem in f.read().split()]


with hexaly.optimizer.HexalyOptimizer() as optimizer:
    #
    # Read instance data
    #
    file_it = iter(read_elem(sys.argv[1]))

    # The input files follow the TSPLib "explicit" format
    for pch in file_it:
        if pch == "N:":
            nb_cities = int(next(file_it))
        if pch == "Distance:[":
            # Distance from i to j
            dist_matrix_data = [[int(next(file_it)) for i in range(nb_cities)]
                                for j in range(nb_cities)]
        if pch == "Demand:":
            next(file_it)
            # Vector representing the demand in terms of weight for each city
            item_weight_data = [int(next(file_it)) for i in range(nb_cities)]
            # Initial weight of the vehicle transporting all the items to be delivered
            total_weight = sum(item_weight_data)
        if pch == "Draft:":
            next(file_it)
            break

    # Vector used to store the weight (draft) limit for each city
    weight_limit_data = [int(next(file_it)) for i in range(nb_cities)]

    #
    # Declare the optimization model
    #
    model = optimizer.model

    # A list variable: cities[i] is the index of the ith city in the tour
    cities = model.list(nb_cities)

    # All cities must be visited
    model.constraint(model.count(cities) == nb_cities)

    # Create Hexaly arrays for all vector data in order to be able to access them with "at" operator.
    dist_matrix = model.array(dist_matrix_data)
    item_weight = model.array(item_weight_data)
    weight_limit = model.array(weight_limit_data)

    # Compute the weight of the vehicle along the tour
    weight_lambda = model.lambda_function(lambda i, prev:
                                          model.iif(i == 0, total_weight, prev - item_weight[cities[i-1]]))
    weight = model.array(model.range(0, nb_cities), weight_lambda, 0)

    # At each step, the vehicle's weight must not exceed the weight limit allowed for the city.
    # N.B: Setting a constraint over all terms of an array is done with the 'and' operator:
    weight_constraint_lambda = model.lambda_function(lambda i:
                                                     weight[i] <= weight_limit[cities[i]])
    model.constraint(model.and_(model.range(0, nb_cities), weight_constraint_lambda))

    # Minimize the total distance
    dist_lambda = model.lambda_function(lambda i:
                                        model.at(dist_matrix, cities[i - 1], cities[i]))
    obj = model.sum(model.range(1, nb_cities), dist_lambda) \
        + model.at(dist_matrix, cities[nb_cities - 1], cities[0])
    model.minimize(obj)

    model.close()

    # Parameterize the optimizer
    if len(sys.argv) >= 4:
        optimizer.param.time_limit = int(sys.argv[3])
    else:
        optimizer.param.time_limit = 5

    optimizer.solve()

    #
    # Write the solution in a file
    #
    if len(sys.argv) >= 3:
        # Write the solution in a file
        with open(sys.argv[2], 'w') as f:
            f.write("%d\n" % obj.value)
            for c in cities.value:
                f.write("%d " % c)
            f.write("\n")
